MISSING VALUES HANDLING

In [3]:
import numpy as np
import pandas as pd
from sklearn import ensemble
import scripts_for_E5 as s
from ydata_profiling import ProfileReport

In [4]:
# Load the clustered database
db = pd.read_csv('db_clustered.csv')

In [5]:
# Convert data types
bool_cols = ['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti', 'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio']
db[bool_cols] = db[bool_cols].astype("boolean")

db["Codice Via"] = db["Codice Via"].astype("Int64")
db["ZD"] = db["ZD"].astype("Int64")

In [6]:
# Generate the profiling report
PROFILE = ProfileReport(db, title="Profiling Report").to_file("db_REPORT_after_wrangling.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:01<00:00, 12.91it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# Drop columns because missing too many values
db = db.drop(["Insegna"], axis = 1)
db = db.drop(["Resto Ubicazione"], axis = 1)
db = db.drop(["Superficie Tabelle Speciali"], axis = 1)

In [8]:
# Reorder columns
db = db[['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti',
       'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio',
       'Tipo Via', 'Via', 'Civico', 'Codice Via', 'ZD', 'Isolato', 'Accesso', 
       'Settore Storico Cf Preval', 'Superficie Vendita', 'Superficie Altri Usi', 'Superficie Totale']]

In [10]:
# Replace 0 with NaN in 'Superficie Vendita' to apply imputation later
db['Superficie Vendita'] = db['Superficie Vendita'].replace(0,np.nan)

In [11]:
# Impute missing values with Random Forest
COLS = ['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti',
       'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio',
       'Tipo Via', 'Via', 'Civico', 'Codice Via', 'ZD', 'Isolato', 'Accesso', 
       'Settore Storico Cf Preval', 'Superficie Vendita', 'Superficie Altri Usi']

missing_columns = ["Alimentare", "Non Alimentare", "Tabella Speciale Carburanti",
       "Tabella Speciale Farmacie", "Tabella Speciale Monopolio", "Superficie Vendita", "Superficie Altri Usi"]

IMP_DATA = pd.DataFrame(columns = ["IMP" + name for name in missing_columns])

CAT = list(db.select_dtypes(include=['boolean','object']).columns)
NUM = list(db.select_dtypes(include=['int64','float64']).columns)

for feature in missing_columns:

    db[feature + '_imp'] = db[feature]

    if feature in NUM:
      db.loc[db[feature].isnull(), feature + '_imp'] = db[feature].median()
    elif feature in CAT:
      db.loc[db[feature].isnull(), feature + '_imp'] = db[feature].mode()[0]

for feature in missing_columns:

    IMP_DATA["IMP" + feature] = db[feature]
    parameters = list(set(db.columns) - set(missing_columns) - {feature + '_imp'})

    if feature in NUM:
      model = ensemble.RandomForestRegressor()
    if feature in CAT:
      model = ensemble.RandomForestClassifier()

    X = s.encoding_categorical_variables(db[parameters])

    model.fit(X = X.loc[db[feature].notnull()], y = db[feature + '_imp'].loc[db[feature].notnull()])
    model_predicted = model.predict(X.loc[db[feature].isnull()])

    print("IMP" + feature + " successfully imputed")
    IMP_DATA.loc[db[feature].isnull(), "IMP" + feature] = model_predicted

IMPAlimentare successfully imputed
IMPNon Alimentare successfully imputed
IMPTabella Speciale Carburanti successfully imputed
IMPTabella Speciale Farmacie successfully imputed
IMPTabella Speciale Monopolio successfully imputed
IMPSuperficie Vendita successfully imputed
IMPSuperficie Altri Usi successfully imputed


In [12]:
# Drop original columns with missing data
db = db.drop(
    [
        "Alimentare",
        "Non Alimentare",
        "Tabella Speciale Carburanti",
        "Tabella Speciale Farmacie",
        "Tabella Speciale Monopolio",
        "Superficie Vendita",
        "Superficie Altri Usi",
    ],
    axis=1
)

# Rename imputed columns
db = db.rename(columns={
    "Alimentare_imp": "Alimentare",
    "Non Alimentare_imp": "Non Alimentare",
    "Tabella Speciale Carburanti_imp": "Tabella Speciale Carburanti",
    "Tabella Speciale Farmacie_imp": "Tabella Speciale Farmacie",
    "Tabella Speciale Monopolio_imp": "Tabella Speciale Monopolio",
    "Superficie Vendita_imp": "Superficie Vendita",
    "Superficie Altri Usi_imp": "Superficie Altri Usi"
})

# Reorder columns
db = db[['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti',
       'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio',
       'Tipo Via', 'Via', 'Civico', 'Codice Via', 'ZD', 'Isolato', 'Accesso', 
       'Settore Storico Cf Preval', 'Superficie Vendita', 'Superficie Altri Usi', 
       'Superficie Totale']]

In [13]:
# Fill categorical columns with "missing"
db["Tipo Via"] = db["Tipo Via"].fillna("missing")
db["Via"] = db["Via"].fillna("missing")
db["Civico"] = db["Civico"].fillna("missing")
db["Accesso"] = db["Accesso"].fillna("missing")

In [14]:
# Fill numerical columns with "0"
db["Isolato"] = db["Isolato"].fillna(0)
db["ZD"] = db["ZD"].fillna(0)
db["Codice Via"] = db["Codice Via"].fillna(0)

In [15]:
# Compute Superficie Totale as sum of Superficie Vendita and Superficie Altri Usi
db["Superficie Totale"] = db[
    ["Superficie Vendita", "Superficie Altri Usi"]
].sum(axis=1)

In [16]:
# Generate the profiling report
PROFILE = ProfileReport(db, title="Profiling Report").to_file("db_REPORT_after_randomForest.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 41.20it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
# Save the database after handling missing values
db.to_csv('db_missing_values_handled.csv')